# Laborator: Sistem RAG local (Retrieval-Augmented Generation)
### FAISS + `sentence-transformers` + model LLM local prin Ollama

**Autor:** Iatco Marcel

**Structura notebook-ului**

0. Instalarea dependențelor
1. Introducere și încărcarea datelor
2. Preprocesare și chunking
3. Embeddings și index vectorial FAISS
4. Generarea răspunsurilor cu RAG (inclusiv verificarea Ollama)
5. Testare și evaluare
6. Concluzii

> Notebook-ul se rulează **de sus în jos**. Singurul pas manual este pornirea Ollama (secțiunea 4).
> Nu se folosesc API-uri plătite și nici chei API: totul rulează local.

## 0. Instalarea dependențelor

Se rulează o singură dată. `%pip` instalează pachetele **în mediul kernel-ului curent** (de exemplu, mediul conda selectat în PyCharm), nu în alt Python de pe calculator.

| Pachet | Rol |
|---|---|
| `sentence-transformers` | modelul de embeddings `all-MiniLM-L6-v2` (instalează și PyTorch) |
| `faiss-cpu` | indexul vectorial pentru căutarea de similaritate |
| `pandas`, `numpy` | prelucrarea datelor |
| `requests` | descărcarea datasetului și comunicarea cu Ollama (API HTTP local) |

Dacă un pachet abia a fost instalat și importul din celula următoare dă eroare, reporniți kernel-ul (*Restart Kernel*) și rulați din nou.

In [1]:
%pip install -q sentence-transformers faiss-cpu pandas numpy requests

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [2]:
import os, re, json, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# Ascunde un avertisment inofensiv al Hugging Face pe Windows (legat de symlink-uri în cache)
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

pd.set_option("display.max_colwidth", 120)
print("Biblioteci de bază importate.")

Biblioteci de bază importate.


## 1. Introducere și încărcarea datelor

### Ce este RAG?
Un LLM răspunde doar din ce a „memorat” la antrenare, iar uneori inventează (*halucinează*). **RAG** adaugă un pas de căutare înaintea generării:

```
întrebare ──► [Retriever: caută fragmentele relevante în baza de cunoștințe] ──► context
                                                                              │
întrebare + context ──► [LLM local (Ollama)] ──► răspuns bazat pe context ◄───┘
```

Astfel, modelul primește informația relevantă direct în prompt și i se cere să răspundă **numai** pe baza ei.

### Datasetul: SQuAD 2.0 (setul *dev*)
- **Sursă:** Stanford Question Answering Dataset 2.0 (Rajpurkar et al., 2018), licență CC BY-SA 4.0, descărcat din repository-ul public al autorilor de pe GitHub (fără autentificare, ~4,4 MB).
- **Conținut:** paragrafe din 35 de articole Wikipedia în limba engleză, plus întrebări scrise de oameni pentru fiecare paragraf.
- **De ce e potrivit pentru RAG:** fiecare întrebare are răspunsul corect cunoscut și paragraful-sursă cunoscut, deci putem **măsura obiectiv** calitatea căutării. În plus, SQuAD 2.0 conține întrebări marcate explicit ca **fără răspuns** (`is_impossible = True`), utile pentru a testa dacă sistemul refuză corect să răspundă.

**Câmpurile folosite**

| Câmp | Semnificație |
|---|---|
| `title` | titlul articolului Wikipedia |
| `context` | textul paragrafului (baza de cunoștințe) |
| `question` | întrebarea |
| `answers` | răspunsurile corecte (fragmente exacte din paragraf) |
| `is_impossible` | `True` dacă paragraful nu conține răspunsul |

**Subset:** folosim 10 articole din domenii diferite (istorie, fizică, chimie, biologie, geografie, matematică), adică câteva sute de paragrafe. Lista se poate modifica în `ARTICLES`.

In [3]:
# ---- Configurare generală ----
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
DATA_FILE = DATA_DIR / "squad_dev_v2.0.json"

# Primul URL: repository-ul oficial al autorilor pe GitHub; al doilea: site-ul oficial SQuAD (rezervă)
DATA_URLS = [
    "https://raw.githubusercontent.com/rajpurkar/SQuAD-explorer/master/dataset/dev-v2.0.json",
    "https://rajpurkar.github.io/SQuAD-explorer/dataset/dev-v2.0.json",
]

# Subsetul de articole folosit ca bază de cunoștințe
ARTICLES = [
    "Normans", "Steam_engine", "Oxygen", "Amazon_rainforest", "Black_Death",
    "Immune_system", "Prime_number", "Rhine", "Warsaw", "Force",
]


def download_dataset(urls, dest, timeout=60):
    # Descarcă fișierul o singură dată; la rulările următoare se folosește copia locală.
    if dest.exists() and dest.stat().st_size > 0:
        print(f"Datasetul există deja local: {dest} ({dest.stat().st_size / 1e6:.1f} MB)")
        return
    errors = []
    for url in urls:
        try:
            print(f"Descarc: {url}")
            r = requests.get(url, timeout=timeout)
            r.raise_for_status()
            json.loads(r.content.decode("utf-8"))      # verificăm că am primit JSON valid
            dest.write_bytes(r.content)
            print(f"Salvat în {dest} ({len(r.content) / 1e6:.1f} MB)")
            return
        except Exception as e:
            errors.append(f"  - {url}\n    {type(e).__name__}: {e}")
    raise RuntimeError(
        "Nu am putut descărca datasetul de la niciuna dintre surse:\n" + "\n".join(errors) +
        "\n\nVerificați conexiunea la internet (sau proxy/firewall). Alternativ, descărcați manual "
        f"fișierul de la primul URL într-un browser și salvați-l ca: {dest.resolve()}"
    )


download_dataset(DATA_URLS, DATA_FILE)

# encoding="utf-8" este important pe Windows (altfel se folosește implicit cp1252)
with open(DATA_FILE, encoding="utf-8") as f:
    squad = json.load(f)["data"]

# Transformăm structura imbricată JSON în două tabele: paragrafe și întrebări
paragraph_rows, question_rows = [], []
for article in squad:
    if article["title"] not in ARTICLES:
        continue
    for p_idx, par in enumerate(article["paragraphs"]):
        pid = f'{article["title"]}#{p_idx}'
        paragraph_rows.append({"paragraph_id": pid, "title": article["title"], "context": par["context"]})
        for qa in par["qas"]:
            question_rows.append({
                "paragraph_id": pid,
                "question": qa["question"].strip(),
                "answers": sorted({a["text"] for a in qa["answers"]}),
                "is_impossible": qa["is_impossible"],
            })

paragraphs_df = pd.DataFrame(paragraph_rows)
questions_df = pd.DataFrame(question_rows)

missing = set(ARTICLES) - set(paragraphs_df["title"])
if missing:
    print("Atenție, articole negăsite în dataset:", missing)

print(f"Articole: {paragraphs_df['title'].nunique()} | paragrafe: {len(paragraphs_df)} | "
      f"întrebări: {len(questions_df)} (din care fără răspuns: {questions_df['is_impossible'].sum()})")
display(paragraphs_df.groupby("title").size().rename("nr_paragrafe").to_frame().T)
display(questions_df.head(3))

Descarc: https://raw.githubusercontent.com/rajpurkar/SQuAD-explorer/master/dataset/dev-v2.0.json
Salvat în data\squad_dev_v2.0.json (4.4 MB)
Articole: 10 | paragrafe: 389 | întrebări: 3697 (din care fără răspuns: 1752)


title,Amazon_rainforest,Black_Death,Force,Immune_system,Normans,Oxygen,Prime_number,Rhine,Steam_engine,Warsaw
nr_paragrafe,21,23,44,49,39,43,31,44,46,49


,paragraph_id,question,answers,is_impossible
0,Normans#0,In what country is Normandy located?,[France],False
1,Normans#0,When were the Normans in Normandy?,"[10th and 11th centuries, in the 10th and 11th centuries]",False
2,Normans#0,From which countries did the Norse originate?,"[Denmark, Iceland and Norway]",False


## 2. Preprocesare și chunking

**Curățare.** Textele SQuAD sunt deja destul de curate, așa că facem doar o normalizare minimă: normalizare Unicode (NFKC), înlocuirea salturilor de rând cu spații și eliminarea spațiilor multiple. Nu eliminăm punctuația și nu transformăm în litere mici, deoarece modelul de embeddings lucrează cu propoziții naturale.

**Chunking.** Un fragment (*chunk*) trebuie să fie suficient de scurt pentru a fi relevant și pentru a încăpea în modelul de embeddings (`all-MiniLM-L6-v2` trunchiază textul după 256 de tokeni, adică aproximativ 180–190 de cuvinte în engleză). Strategia aleasă:
- paragrafele de cel mult `MAX_WORDS` cuvinte rămân un singur fragment;
- paragrafele mai lungi sunt împărțite **la granița propozițiilor**, cu o **suprapunere de o propoziție** între fragmente consecutive, ca să nu se piardă contextul la tăietură.

Fiecare fragment păstrează `paragraph_id`, care va fi folosit la evaluare.

In [4]:
MAX_WORDS = 150       # lungimea maximă a unui fragment (în cuvinte)
OVERLAP_SENT = 1      # câte propoziții se repetă între fragmente consecutive


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_sentences(text: str) -> list:
    # Împărțire simplă: după . ! ? urmate de spațiu și de o literă mare, cifră sau ghilimele.
    return [s for s in re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"“(])', text) if s]


def chunk_text(text: str, max_words: int = MAX_WORDS, overlap: int = OVERLAP_SENT) -> list:
    if len(text.split()) <= max_words:
        return [text]
    sentences = split_sentences(text)
    chunks, current = [], []
    for sent in sentences:
        if current and len(" ".join(current + [sent]).split()) > max_words:
            chunks.append(" ".join(current))
            current = current[-overlap:] if overlap else []   # suprapunere
            # dacă suprapunerea + propoziția nouă ar depăși limita, renunțăm la suprapunere
            if len(" ".join(current + [sent]).split()) > max_words:
                current = []
        current.append(sent)
    if current:
        chunks.append(" ".join(current))
    return chunks


chunk_rows = []
for row in paragraphs_df.itertuples():
    for i, ch in enumerate(chunk_text(clean_text(row.context))):
        chunk_rows.append({"chunk_id": len(chunk_rows), "paragraph_id": row.paragraph_id,
                           "title": row.title, "part": i, "text": ch, "n_words": len(ch.split())})

chunks_df = pd.DataFrame(chunk_rows)
dup = chunks_df["text"].duplicated().sum()
chunks_df = chunks_df.drop_duplicates("text").reset_index(drop=True)
chunks_df["chunk_id"] = chunks_df.index      # chunk_id = poziția în indexul FAISS

print(f"Paragrafe: {len(paragraphs_df)} -> fragmente: {len(chunks_df)} (duplicate eliminate: {dup})")
print("Paragrafe împărțite în mai multe fragmente:", (chunks_df.groupby('paragraph_id').size() > 1).sum())
print("Statistici lungime fragmente (cuvinte):")
display(chunks_df["n_words"].describe().round(1).to_frame().T)

# Exemplu: un paragraf lung împărțit în fragmente (se vede suprapunerea)
example_pid = chunks_df.loc[chunks_df["part"] == 1, "paragraph_id"].iloc[0]
for t in chunks_df.loc[chunks_df["paragraph_id"] == example_pid, "text"]:
    print(f"\n[{example_pid}] ({len(t.split())} cuvinte)\n{t}")

Paragrafe: 389 -> fragmente: 465 (duplicate eliminate: 0)
Paragrafe împărțite în mai multe fragmente: 72
Statistici lungime fragmente (cuvinte):


,count,mean,std,min,25%,50%,75%,max
n_words,465.0,105.7,28.7,25.0,88.0,106.0,129.0,150.0



[Normans#1] (130 cuvinte)
The Norman dynasty had a major political, cultural and military impact on medieval Europe and even the Near East. The Normans were famed for their martial spirit and eventually for their Christian piety, becoming exponents of the Catholic orthodoxy into which they assimilated. They adopted the Gallo-Romance language of the Frankish land they settled, their dialect becoming known as Norman, Normaund or Norman French, an important literary language. The Duchy of Normandy, which they formed by treaty with the French crown, was a great fief of medieval France, and under Richard I of Normandy was forged into a cohesive and formidable principality in feudal tenure. The Normans are noted both for their culture, such as their unique Romanesque architecture and musical traditions, and for their significant military accomplishments and innovations.

[Normans#1] (121 cuvinte)
The Normans are noted both for their culture, such as their unique Romanesque architecture and 

## 3. Embeddings și index vectorial FAISS

- **Embeddings:** fiecare fragment este transformat într-un vector de 384 de dimensiuni cu `all-MiniLM-L6-v2`, un model mic, rapid pe CPU și antrenat pentru similaritate semantică în **limba engleză**. La prima rulare modelul se descarcă automat de pe Hugging Face (~90 MB), apoi este păstrat în cache.
- **Index:** vectorii sunt **normalizați** (lungime 1), deci produsul scalar este egal cu similaritatea cosinus. Folosim `faiss.IndexFlatIP`, adică o căutare **exactă** (fără aproximări), potrivită pentru câteva sute sau mii de vectori.
- **Retriever:** întrebarea este transformată în vector cu același model, iar FAISS întoarce cele mai apropiate `k` fragmente.

In [5]:
import faiss
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

try:
    embedder = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
except Exception as e:
    raise RuntimeError(
        f"Nu am putut încărca modelul de embeddings '{EMBED_MODEL_NAME}'.\n"
        "La prima rulare, modelul se descarcă de pe huggingface.co: verificați conexiunea la internet "
        "(sau proxy/firewall) și rulați din nou celula.\n"
        f"Eroare originală: {type(e).__name__}: {e}"
    ) from e

t0 = time.perf_counter()
chunk_embeddings = embedder.encode(
    chunks_df["text"].tolist(), batch_size=64, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True,
).astype("float32")
embed_seconds = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
index.add(chunk_embeddings)

print(f"Embeddings: {chunk_embeddings.shape} | timp măsurat: {embed_seconds:.1f} s | vectori în index: {index.ntotal}")

C:\Users\marce\miniconda3\envs\rag-lab\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 8/8 [00:10<00:00,  1.28s/it]

Embeddings: (465, 384) | timp măsurat: 10.3 s | vectori în index: 465


In [6]:
def retrieve(query: str, k: int = 4) -> pd.DataFrame:
    # Întoarce cele mai relevante k fragmente pentru întrebare, împreună cu scorul cosinus.
    q_vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_vec, k)
    result = chunks_df.iloc[ids[0]][["chunk_id", "paragraph_id", "title", "text"]].copy()
    result.insert(0, "score", scores[0].round(3))
    return result.reset_index(drop=True)


# Test rapid al retriever-ului
display(retrieve("Who was the first European to travel the Amazon River?", k=3))

,score,chunk_id,paragraph_id,title,text
0,0.596,155,Amazon_rainforest#6,Amazon_rainforest,The first European to travel the length of the Amazon River was Francisco de Orellana in 1542. The BBC's Unnatural H...
1,0.446,150,Amazon_rainforest#2,Amazon_rainforest,"During the mid-Eocene, it is believed that the drainage basin of the Amazon was split along the middle of the contin..."
2,0.418,154,Amazon_rainforest#5,Amazon_rainforest,"For a long time, it was thought that the Amazon rainforest was only ever sparsely populated, as it was impossible to..."


## 4. Generarea răspunsurilor cu RAG

### 4.1 Verificarea serviciului Ollama și a modelului
Ollama rulează ca serviciu local la adresa `http://localhost:11434`. Celula de mai jos face **o singură** verificare rapidă (timeout de câteva secunde), fără să pornească nimic automat și fără reîncercări:
1. dacă serviciul răspunde;
2. dacă modelul LLM ales (`LLM_MODEL`) este deja descărcat.

**Dacă Ollama nu este pornită (Windows), alegeți una dintre variante:**
- deschideți aplicația **Ollama** din meniul Start (apare apoi o pictogramă lângă ceas, în system tray); **sau**
- deschideți un terminal separat (PowerShell sau Command Prompt) și rulați `ollama serve`, apoi lăsați fereastra deschisă cât timp lucrați.

După pornire, **rulați din nou doar celula de verificare**. Restul notebook-ului (datele, indexul) rămâne în memorie.

**Modelul ales:** `llama3.2:3b` (~2 GB), un model mic care rulează acceptabil pe CPU. Puteți schimba `LLM_MODEL` cu alt model din biblioteca Ollama (de exemplu `qwen2.5:3b`).

In [7]:
OLLAMA_URL = "http://localhost:11434"
LLM_MODEL = "llama3.2:3b"


def _full_name(name: str) -> str:
    # Ollama adaugă implicit eticheta ":latest" dacă nu este specificată
    return name if ":" in name else name + ":latest"


def check_ollama(model: str = LLM_MODEL, url: str = OLLAMA_URL, timeout: float = 3.0) -> bool:
    # O singură verificare: (1) serviciul răspunde, (2) modelul este descărcat. Nu pornește nimic.
    try:
        r = requests.get(f"{url}/api/tags", timeout=timeout)
        r.raise_for_status()
    except requests.exceptions.ConnectionError:
        print(f"❌ Serviciul Ollama NU răspunde la {url}.\n")
        print("Pornire manuală în Windows (alegeți una dintre variante):")
        print("  1) Deschideți aplicația 'Ollama' din meniul Start (pictograma apare în system tray), SAU")
        print("  2) Într-un terminal separat (PowerShell / cmd) rulați:  ollama serve")
        print("     și lăsați fereastra deschisă.")
        print("\nApoi rulați din nou ACEASTĂ celulă.")
        print("(Dacă 'ollama serve' afișează o eroare de tip 'address already in use', serviciul rulează deja.)")
        return False
    except requests.exceptions.Timeout:
        print(f"❌ Ollama nu a răspuns în {timeout} s la {url}. Verificați dacă serviciul pornește corect și reîncercați.")
        return False
    except requests.exceptions.RequestException as e:
        print(f"❌ Eroare la comunicarea cu Ollama: {type(e).__name__}: {e}")
        return False

    installed = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Serviciul Ollama rulează la {url}.")
    print("   Modele descărcate local:", ", ".join(installed) if installed else "(niciunul)")

    if _full_name(model) not in [_full_name(m) for m in installed]:
        print(f"\n❌ Modelul '{model}' NU este descărcat.")
        print("Într-un terminal (PowerShell / cmd) rulați comanda:\n")
        print(f"    ollama pull {model}\n")
        print("Descărcarea se face O SINGURĂ DATĂ (modelul rămâne apoi pe disc). Pentru llama3.2:3b sunt ~2 GB.")
        print("Dacă descărcarea eșuează, verificați conexiunea la internet și spațiul liber pe disc, apoi")
        print("rulați din nou comanda (descărcarea se reia de unde a rămas). La final, rulați din nou această celulă.")
        return False

    print(f"✅ Modelul '{model}' este disponibil. Puteți continua.")
    return True


OLLAMA_READY = check_ollama()

✅ Serviciul Ollama rulează la http://localhost:11434.
   Modele descărcate local: llama3.2:3b, gemma3:4b, mistral:7b, llama3.1:8b, qwen3:8b, phi3:3.8b
✅ Modelul 'llama3.2:3b' este disponibil. Puteți continua.


### 4.2 Promptul și apelul către LLM
- **Promptul RAG** conține fragmentele recuperate (numerotate, cu titlul articolului) și instrucțiunea de a răspunde **doar** din context. Dacă informația lipsește, modelul trebuie să răspundă exact cu fraza `I don't know based on the provided context.`; astfel putem detecta automat refuzurile.
- **Promptul fără RAG** conține doar întrebarea și este folosit pentru comparație în secțiunea 5.
- Folosim `temperature = 0` pentru răspunsuri cât mai deterministe și reproductibile.
- Întrebările și contextul sunt în engleză (limba datasetului), deci și răspunsurile sunt cerute în engleză, ca să poată fi comparate cu răspunsurile corecte din SQuAD.

In [8]:
TOP_K = 4
NO_ANSWER = "I don't know based on the provided context."

SYSTEM_RAG = (
    "You are a question-answering assistant. Answer ONLY using the information in the provided context. "
    f"If the context does not contain the answer, reply exactly: \"{NO_ANSWER}\" "
    "Do not use outside knowledge. Answer in English, briefly (one or two sentences)."
)
SYSTEM_NO_RAG = (
    "You are a question-answering assistant. Answer in English, briefly (one or two sentences). "
    "If you do not know the answer, say \"I don't know.\""
)


def build_rag_prompt(question: str, retrieved: pd.DataFrame) -> str:
    context = "\n\n".join(
        f"[{i + 1}] (source: {row.title})\n{row.text}" for i, row in enumerate(retrieved.itertuples())
    )
    return f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"


def ask_llm(system: str, user: str, model: str = LLM_MODEL, timeout: int = 600) -> str:
    # Apel non-streaming către /api/chat; ridică o eroare clară dacă ceva nu merge.
    payload = {
        "model": model,
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
        "stream": False,
        "options": {"temperature": 0, "num_ctx": 4096},
    }
    try:
        r = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=timeout)
    except requests.exceptions.ConnectionError as e:
        raise RuntimeError("Conexiunea la Ollama s-a pierdut. Reporniți Ollama și rulați din nou "
                           "celula de verificare (4.1).") from e
    except requests.exceptions.Timeout as e:
        raise RuntimeError(f"Ollama nu a răspuns în {timeout} s. Pe CPU, prima cerere include și încărcarea "
                           "modelului în memorie; reîncercați sau folosiți un model mai mic.") from e
    if r.status_code != 200:
        try:
            err = r.json().get("error", r.text)
        except ValueError:
            err = r.text
        raise RuntimeError(f"Ollama a întors HTTP {r.status_code}: {err}\n"
                           f"(Dacă modelul lipsește, rulați: ollama pull {model})")
    return r.json()["message"]["content"].strip()


def rag_answer(question: str, k: int = TOP_K) -> dict:
    retrieved = retrieve(question, k)
    t0 = time.perf_counter()
    answer = ask_llm(SYSTEM_RAG, build_rag_prompt(question, retrieved))
    return {"answer": answer, "retrieved": retrieved, "seconds": time.perf_counter() - t0}


def no_rag_answer(question: str) -> dict:
    t0 = time.perf_counter()
    answer = ask_llm(SYSTEM_NO_RAG, f"Question: {question}\nAnswer:")
    return {"answer": answer, "seconds": time.perf_counter() - t0}


# Afișăm un exemplu de prompt complet (nu necesită Ollama)
print(build_rag_prompt("Where is the highest point of the Rhine basin?",
                       retrieve("Where is the highest point of the Rhine basin?", k=2)))

Context:
[1] (source: Rhine)
The Rhine emerges from Lake Constance, flows generally westward, as the Hochrhein, passes the Rhine Falls, and is joined by its major tributary, the river Aare. The Aare more than doubles the Rhine's water discharge, to an average of nearly 1,000 m3/s (35,000 cu ft/s), and provides more than a fifth of the discharge at the Dutch border. The Aare also contains the waters from the 4,274 m (14,022 ft) summit of Finsteraarhorn, the highest point of the Rhine basin. The Rhine roughly forms the German-Swiss border from Lake Constance with the exceptions of the canton of Schaffhausen and parts of the cantons of Zürich and Basel-Stadt, until it turns north at the so-called Rhine knee at Basel, leaving Switzerland.

[2] (source: Rhine)
In the centre of Basel, the first major city in the course of the stream, is located the "Rhine knee"; this is a major bend, where the overall direction of the Rhine changes from West to North. Here the High Rhine ends. Legally, the C

In [9]:
if OLLAMA_READY:
    demo = rag_answer("Where is the highest point of the Rhine basin?")
    print("Răspuns RAG:", demo["answer"])
    print(f"Timp de generare măsurat: {demo['seconds']:.1f} s")
else:
    print("Ollama nu este pregătită: rulați din nou celula de verificare din 4.1 după pornirea serviciului.")

Răspuns RAG: The highest point of the Rhine basin is the summit of Finsteraarhorn, which is 4,274 m (14,022 ft) high.
Timp de generare măsurat: 17.2 s


## 5. Testare și evaluare

### 5.1 Evaluarea căutării (fără LLM)
Pentru întrebările cu răspuns cunoscut verificăm automat dacă retriever-ul aduce fragmentul potrivit. Un rezultat este considerat **corect** dacă fragmentul recuperat provine din **paragraful-sursă** al întrebării **și** conține unul dintre răspunsurile corecte.

- **Hit@k:** procentul de întrebări pentru care cel puțin un fragment corect se află între primele `k` rezultate.
- **MRR** (*Mean Reciprocal Rank*): media valorii 1/poziția primului fragment corect (1 dacă este primul, 0,5 dacă este al doilea, ..., 0 dacă nu apare în top 10).

Folosim un eșantion aleator fix (`random_state=42`) de întrebări cu răspuns, ca rezultatul să fie reproductibil.

In [10]:
N_EVAL = 300
eval_q = questions_df[~questions_df["is_impossible"]].sample(n=N_EVAL, random_state=42).reset_index(drop=True)

q_vecs = embedder.encode(eval_q["question"].tolist(), batch_size=64, convert_to_numpy=True,
                         normalize_embeddings=True).astype("float32")
_, top_ids = index.search(q_vecs, 10)

first_hit_rank = []          # poziția (1..10) a primului fragment corect, sau None
for qi, row in eval_q.iterrows():
    rank = None
    for pos, cid in enumerate(top_ids[qi], start=1):
        ch = chunks_df.iloc[cid]
        if ch["paragraph_id"] == row["paragraph_id"] and any(clean_text(a) in ch["text"] for a in row["answers"]):
            rank = pos
            break
    first_hit_rank.append(rank)

eval_q["first_hit_rank"] = first_hit_rank
retrieval_metrics = {f"Hit@{k}": np.mean([r is not None and r <= k for r in first_hit_rank]) for k in (1, 3, 5, 10)}
retrieval_metrics["MRR@10"] = np.mean([1 / r if r else 0 for r in first_hit_rank])

print(f"Evaluare căutare pe {N_EVAL} întrebări cu răspuns ({len(chunks_df)} fragmente în index):")
display(pd.Series(retrieval_metrics).round(3).to_frame("valoare").T)

print("Exemple de întrebări la care fragmentul corect NU a apărut în top 10:")
display(eval_q[eval_q["first_hit_rank"].isna()][["question", "answers", "paragraph_id"]].head(5))

Evaluare căutare pe 300 întrebări cu răspuns (465 fragmente în index):


,Hit@1,Hit@3,Hit@5,Hit@10,MRR@10
valoare,0.673,0.857,0.917,0.96,0.78


Exemple de întrebări la care fragmentul corect NU a apărut în top 10:


,question,answers,paragraph_id
7,What did the analysis from the sediment deposits indicate?,"[rainfall in the basin during the LGM was lower, rainfall in the basin during the LGM was lower than for the present]",Amazon_rainforest#3
52,Where does the Rhine begin?,"[Old Rhine Bridge, Old Rhine Bridge at Constance, the Old Rhine Bridge at Constance]",Rhine#2
79,Who did the army siege belong to?,[Jani Beg],Black_Death#2
93,What name is given to any prime number larger than 2?,[odd prime],Prime_number#4
99,Who was Robert's son?,[Bohemond],Normans#12


### 5.2 Întrebări de test: RAG vs. fără RAG
Testăm 8 întrebări:
- **6 întrebări cu răspuns** în baza de cunoștințe (preluate din SQuAD, cu răspunsul corect cunoscut);
- **1 întrebare-capcană din SQuAD 2.0** (`is_impossible = True`): „În ce an a fost fondat orașul Kaffa?”. Articolul *Black Death* menționează Kaffa și anul 1347, dar **nu** anul fondării. Un sistem corect ar trebui să refuze să răspundă;
- **1 întrebare din afara domeniului** (Campionatul Mondial de fotbal 2022), care nu are legătură cu cele 10 articole.

Pentru fiecare întrebare afișăm fragmentele recuperate, răspunsul **cu RAG**, răspunsul **fără RAG** (același model, aceeași temperatură) și două verificări automate:
- `contains_gold`: răspunsul conține unul dintre răspunsurile corecte (după normalizare: litere mici, fără punctuație și fără articole). Este o verificare **aproximativă**: un răspuns corect formulat altfel poate fi marcat greșit, deci tabelul trebuie verificat și manual;
- `refused`: modelul a spus că nu știe.

**Despre corectitudinea comparației:** fără RAG, modelul poate ști răspunsul din antrenare (Wikipedia face parte, de regulă, din datele de antrenare). Comparația arată deci în principal dacă răspunsul este **ancorat într-o sursă verificabilă** și dacă modelul refuză atunci când informația lipsește, nu doar dacă „știe mai mult”. Pentru întrebarea din afara domeniului, un răspuns corect fără RAG nu este o greșeală a modelului: este comportamentul așteptat de la un LLM general, în timp ce sistemul RAG trebuie să refuze, deoarece baza lui de cunoștințe nu conține informația.

In [11]:
TEST_QUESTIONS = [
    {"question": "Who was the first European to travel the Amazon River?", "gold": ["Francisco de Orellana"], "type": "cu răspuns"},
    {"question": "Which building in Warsaw was a gift from the Soviet Union?", "gold": ["Palace of Culture and Science"], "type": "cu răspuns"},
    {"question": "As of January 2016, how many digits does the largest known prime consist of?", "gold": ["22,338,618"], "type": "cu răspuns"},
    {"question": "What type of steam engine produces most electricity in the world today?", "gold": ["turbine"], "type": "cu răspuns"},
    {"question": "Where is the highest point of the Rhine basin?", "gold": ["Finsteraarhorn"], "type": "cu răspuns"},
    {"question": "What was Warsaw's first literary cabaret?", "gold": ["Momus"], "type": "cu răspuns"},
    {"question": "In what year was Kaffa founded?", "gold": [], "type": "fără răspuns (SQuAD)"},
    {"question": "Who won the 2022 FIFA World Cup?", "gold": [], "type": "în afara domeniului"},
]


def normalize(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^\w\s]", "", s)                 # elimină punctuația (inclusiv virgulele din numere)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def contains_gold(answer: str, gold: list):
    if not gold:
        return None
    return any(normalize(g) in normalize(answer) for g in gold)


def refused(answer: str) -> bool:
    return "i dont know" in normalize(answer)


test_results = []
if not OLLAMA_READY:
    print("Ollama nu este pregătită: porniți serviciul, rulați din nou celula 4.1, apoi această celulă.")
else:
    for i, tq in enumerate(TEST_QUESTIONS, start=1):
        q = tq["question"]
        rag = rag_answer(q)
        base = no_rag_answer(q)

        print("=" * 100)
        print(f"[{i}] ({tq['type']}) {q}")
        print("Răspuns corect (SQuAD):", tq["gold"] if tq["gold"] else "— (nu există în baza de cunoștințe)")
        print("\nFragmente recuperate:")
        for r in rag["retrieved"].itertuples():
            print(f"  score={r.score:.3f} | {r.paragraph_id} | {r.text[:160]}...")
        print(f"\nCu RAG   ({rag['seconds']:.1f} s): {rag['answer']}")
        print(f"Fără RAG ({base['seconds']:.1f} s): {base['answer']}")

        test_results.append({
            "nr": i, "tip": tq["type"], "question": q,
            "top1_score": rag["retrieved"]["score"].iloc[0],
            "rag_answer": rag["answer"], "no_rag_answer": base["answer"],
            "rag_contains_gold": contains_gold(rag["answer"], tq["gold"]),
            "no_rag_contains_gold": contains_gold(base["answer"], tq["gold"]),
            "rag_refused": refused(rag["answer"]), "no_rag_refused": refused(base["answer"]),
            "rag_s": round(rag["seconds"], 1), "no_rag_s": round(base["seconds"], 1),
        })

results_df = pd.DataFrame(test_results)
if not results_df.empty:
    display(results_df.drop(columns=["rag_answer", "no_rag_answer"]))

[1] (cu răspuns) Who was the first European to travel the Amazon River?
Răspuns corect (SQuAD): ['Francisco de Orellana']

Fragmente recuperate:
  score=0.596 | Amazon_rainforest#6 | The first European to travel the length of the Amazon River was Francisco de Orellana in 1542. The BBC's Unnatural Histories presents evidence that Orellana, ra...
  score=0.446 | Amazon_rainforest#2 | During the mid-Eocene, it is believed that the drainage basin of the Amazon was split along the middle of the continent by the Purus Arch. Water on the eastern ...
  score=0.418 | Amazon_rainforest#5 | For a long time, it was thought that the Amazon rainforest was only ever sparsely populated, as it was impossible to sustain a large population through agricult...
  score=0.402 | Amazon_rainforest#7 | Terra preta (black earth), which is distributed over large areas in the Amazon forest, is now widely accepted as a product of indigenous soil management. The de...

Cu RAG   (14.4 s): Francisco de Orellana was t

,nr,tip,question,top1_score,rag_contains_gold,no_rag_contains_gold,rag_refused,no_rag_refused,rag_s,no_rag_s
0,1,cu răspuns,Who was the first European to travel the Amazon River?,0.596,True,False,False,True,14.4,4.5
1,2,cu răspuns,Which building in Warsaw was a gift from the Soviet Union?,0.671,True,True,False,False,12.7,3.3
2,3,cu răspuns,"As of January 2016, how many digits does the largest known prime consist of?",0.693,True,False,False,False,14.1,3.6
3,4,cu răspuns,What type of steam engine produces most electricity in the world today?,0.714,True,True,False,True,13.3,5.0
4,5,cu răspuns,Where is the highest point of the Rhine basin?,0.673,True,False,False,False,4.0,4.8
5,6,cu răspuns,What was Warsaw's first literary cabaret?,0.527,True,False,False,True,12.4,2.6
6,7,fără răspuns (SQuAD),In what year was Kaffa founded?,0.238,None,None,True,True,14.5,2.5
7,8,în afara domeniului,Who won the 2022 FIFA World Cup?,0.214,None,None,True,False,11.8,4.1


**Observație despre scorul de similaritate:** coloana `top1_score` arată similaritatea cosinus a celui mai bun fragment. Comparați valorile pentru întrebările din afara domeniului cu cele pentru întrebările cu răspuns: retriever-ul întoarce **întotdeauna** `k` fragmente, chiar și atunci când niciunul nu este relevant. De aceea instrucțiunea de refuz din prompt este necesară.

## 6. Concluzii

Celula următoare rezumă **rezultatele efective ale rulării curente**. Concluziile de mai jos trebuie completate pe baza acestor cifre și a răspunsurilor afișate în secțiunea 5.2; ele nu sunt scrise dinainte.

In [12]:
print("=== Rezumatul rulării ===")
print(f"Bază de cunoștințe: {paragraphs_df['title'].nunique()} articole, {len(paragraphs_df)} paragrafe, {len(chunks_df)} fragmente")
print(f"Embeddings: {EMBED_MODEL_NAME}, timp de indexare {embed_seconds:.1f} s")
print("Căutare (eșantion de", N_EVAL, "întrebări):", {k: round(float(v), 3) for k, v in retrieval_metrics.items()})

if results_df.empty:
    print("\nSecțiunea 5.2 nu a fost rulată cu Ollama: nu există rezultate de generare.")
else:
    ans = results_df[results_df["tip"] == "cu răspuns"]
    neg = results_df[results_df["tip"] != "cu răspuns"]
    print(f"\nLLM: {LLM_MODEL}")
    print(f"Întrebări cu răspuns ({len(ans)}): conțin răspunsul corect -> cu RAG {int(ans['rag_contains_gold'].sum())}/{len(ans)}, "
          f"fără RAG {int(ans['no_rag_contains_gold'].sum())}/{len(ans)}  (verificare automată aproximativă)")
    print(f"Întrebări fără răspuns în bază ({len(neg)}): refuz -> cu RAG {int(neg['rag_refused'].sum())}/{len(neg)}, "
          f"fără RAG {int(neg['no_rag_refused'].sum())}/{len(neg)}")
    print(f"Timp mediu de generare: cu RAG {results_df['rag_s'].mean():.1f} s, fără RAG {results_df['no_rag_s'].mean():.1f} s")

=== Rezumatul rulării ===
Bază de cunoștințe: 10 articole, 389 paragrafe, 465 fragmente
Embeddings: sentence-transformers/all-MiniLM-L6-v2, timp de indexare 10.3 s
Căutare (eșantion de 300 întrebări): {'Hit@1': 0.673, 'Hit@3': 0.857, 'Hit@5': 0.917, 'Hit@10': 0.96, 'MRR@10': 0.78}

LLM: llama3.2:3b
Întrebări cu răspuns (6): conțin răspunsul corect -> cu RAG 6/6, fără RAG 2/6  (verificare automată aproximativă)
Întrebări fără răspuns în bază (2): refuz -> cu RAG 2/2, fără RAG 1/2
Timp mediu de generare: cu RAG 12.2 s, fără RAG 3.8 s


### Ce face sistemul (prin construcție)
- Baza de cunoștințe este descărcată, curățată, împărțită în fragmente și indexată complet local; după prima descărcare a datelor și a modelelor nu mai este nevoie de internet.
- Răspunsurile cu RAG sunt **trasabile**: pentru fiecare răspuns se văd fragmentele (și paragrafele SQuAD) din care provine.
- Calitatea căutării este măsurată **obiectiv** (Hit@k, MRR), separat de calitatea generării.

### Limite cunoscute ale sistemului
- `all-MiniLM-L6-v2` este antrenat pentru engleză; întrebările în română ar fi căutate mult mai slab. Pentru întrebări în română se poate folosi un model multilingv (de exemplu `paraphrase-multilingual-MiniLM-L12-v2`), cu re-evaluarea metricilor.
- Retriever-ul întoarce mereu `k` fragmente, chiar dacă nu sunt relevante; refuzul depinde în întregime de faptul că LLM-ul respectă instrucțiunea din prompt.
- Un model mic (3B parametri) poate ignora contextul, poate răspunde prea pe larg sau poate folosi cunoștințe proprii.
- `contains_gold` este o metrică aproximativă: poate marca greșit un răspuns corect formulat altfel. Evaluarea pe 8 întrebări este ilustrativă, nu statistic semnificativă.
- Nu există reranking, căutare hibridă (BM25 + vectori) sau prag de similaritate: acestea sunt extinderi posibile.

### Observații din rularea mea *(de completat după rulare, pe baza celulei de rezumat și a secțiunii 5.2)*
- Calitatea căutării (Hit@1 / Hit@5 / MRR): ...
- Întrebări cu răspuns: cu RAG vs. fără RAG: ...
- Întrebarea-capcană „Kaffa” și întrebarea din afara domeniului: a refuzat sistemul RAG? Ce a răspuns modelul fără RAG? ...
- Valorile `top1_score` pentru întrebările fără răspuns, comparativ cu cele cu răspuns: ...
- Timpi de răspuns pe calculatorul meu (CPU): ...